In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
from loaders import loader_XA_to_NA
import importlib
importlib.reload(loader_XA_to_NA)
import tifffile
import SimpleITK
from loaders import stitcher_XA
from cil.framework import ImageGeometry, ImageData
from cil.optimisation.operators import GradientOperator
from cil.framework import AcquisitionGeometry, AcquisitionData, ImageGeometry, ImageData, BlockDataContainer
from cil.plugins.astra.operators import ProjectionOperator
from helpers import module_auxiliary as ma
from cil.optimisation.functions import L2NormSquared, LeastSquares

ModuleNotFoundError: No module named 'loader_XA_to_NA'

In [ ]:
tifffile.imread()

In [ ]:
xray_slices = range(600,700)
neutron_slices = range(1250,1260)
output_volume = [[1250,1260],[None], [None]]
reg = loader_XA_to_NA.load_subset_of_registered_data(dataset_XA='tv', dataset_NA='tv' , h=1, xray_slices = xray_slices,
    neutron_slices = neutron_slices, output_volume = output_volume)

xray_slices = range(600,700)
neutron_slices = range(1250,1260)
output_volume = [[1250,1260],[None], [None]]
reg_fbp = loader_XA_to_NA.load_subset_of_registered_data(dataset_XA='tv', dataset_NA='fbp' , h=1, xray_slices = xray_slices,
    neutron_slices = neutron_slices, output_volume = output_volume)

In [3]:
XA = sitk.GetArrayFromImage(reg.moving).astype(np.float32)
NA = sitk.GetArrayFromImage(reg.fixed).astype(np.float32)
NA_fbp = sitk.GetArrayFromImage(reg_fbp.fixed).astype(np.float32)

In [4]:
path_cache_sino = '/dtu-compute/msaca/sliceA_neutron_psi/output/cache/sino_#####.tiff'
batch = range(1250,1260)
read_path = ma.generate_paths(path_cache_sino, batch)
image = tifffile.imread(read_path[0])
N_pixels = np.shape(image)[1]
N_angles = 1126
angles = np.linspace(0, 360, N_angles, endpoint=True, dtype=np.float32)
data_batch = np.empty((len(batch), N_angles, N_pixels), dtype = np.float32)

for i in range(len(batch)):
    data_batch[i] = tifffile.imread(read_path[i])
N_batch_slices = np.shape(data_batch)[0]

ag_batch = AcquisitionGeometry.create_Parallel3D(detector_position=[0,N_pixels//2,0])\
                            .set_angles(angles)\
                            .set_panel((N_pixels,N_batch_slices), pixel_size=(1,1))\
                            .set_labels(labels=('vertical','angle','horizontal'))

data_batch = AcquisitionData(data_batch, geometry=ag_batch)
data_batch.reorder('astra')

ag_batch.set_angles(ag_batch.angles, initial_angle=+10)
ig_batch_ = ag_batch.get_ImageGeometry()
from cil.processors import Slicer
roi = {'horizontal_y':(820,1180,1), 'horizontal_x':(None,1600,1)}
processor = Slicer(roi)
processor.set_input(ig_batch_)
ig_batch = processor.get_output()
device = 'gpu'

XA_ = ImageData(XA, geometry=ig_batch)


In [ ]:
initial = ig_batch.allocate(0)
A = ProjectionOperator(ig_batch,ag_batch,device)
b = data_batch
print(b)
normA = A.norm()
sigma = 1./normA
tau = 1./normA


alpha_dtv = 0.000001
eta = 0.01
N_iter = 20

In [6]:
def xi_vector_field(image,eta):
    G = GradientOperator(ig_batch)
    numerator = G.direct(image)
    denominator = np.sqrt(eta**2 + numerator.get_item(0)**2 + numerator.get_item(1)**2)
    xi = numerator/denominator

    return (xi.get_item(0)**2 + xi.get_item(1)**2).sqrt()
grad_im = xi_vector_field(XA_, eta)

In [ ]:
from cil.plugins.ccpi_regularisation.functions import FGP_dTV, FGP_TV
from cil.optimisation.algorithms import PDHG, FISTA
alpha_sweep = [100, 300, 700, 2000]


In [ ]:

N_iter = 25
for alpha_TV in alpha_sweep:


    G = alpha_TV * FGP_dTV(reference=XA_, eta=eta, device='gpu', nonnegativity=False)
    F = LeastSquares(A,b)
    reconstructor = FISTA(f=F, g=G, initial=initial)

    reconstructor.run(N_iter, verbose = 1)
    recon_slice_dTV = reconstructor.solution.copy().as_array().astype(np.float32)


    G = alpha_TV*FGP_TV(device='gpu',nonnegativity=False)
    F = LeastSquares(A,b)
    reconstructor = FISTA(f=F, g=G, initial=initial)

    reconstructor.run(N_iter, verbose = 1)
    recon_slice_TV = reconstructor.solution.copy().as_array().astype(np.float32)


    slices = 5
    y0,y1 = 100, 300
    x0, x1 = 400, 800
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
    fig.patch.set_facecolor('black')  # Set the background color to black
    im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,0].set_title('FBP, NA recon', color='red')
    axes[0,0].axis('off')
    im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,1].set_title('Old TV, NA recon', color='red')
    axes[0,1].axis('off')
    im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,0].axis('off')
    im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,1].axis('off')
    im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
    axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
    axes[2,0].axis('off')
    im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
    axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
    axes[2,1].axis('off')
    fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)

    slices = 5
    y0,y1 = 100, 330
    x0, x1 = 800, 1100
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
    fig.patch.set_facecolor('black')  # Set the background color to black
    im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,0].set_title('FBP, NA recon', color='red')
    axes[0,0].axis('off')
    im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,1].set_title('Old TV, NA recon', color='red')
    axes[0,1].axis('off')
    im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,0].axis('off')
    im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,1].axis('off')
    im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
    axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
    axes[2,0].axis('off')
    im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
    axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
    axes[2,1].axis('off')
    fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)



In [ ]:
N_iter = 50
for alpha_TV in alpha_sweep:


    G = alpha_TV * FGP_dTV(reference=XA_, eta=eta, device='gpu', nonnegativity=False)
    F = LeastSquares(A,b)
    reconstructor = FISTA(f=F, g=G, initial=initial)

    reconstructor.run(N_iter, verbose = 1)
    recon_slice_dTV = reconstructor.solution.copy().as_array().astype(np.float32)


    G = alpha_TV*FGP_TV(device='gpu',nonnegativity=False)
    F = LeastSquares(A,b)
    reconstructor = FISTA(f=F, g=G, initial=initial)

    reconstructor.run(N_iter, verbose = 1)
    recon_slice_TV = reconstructor.solution.copy().as_array().astype(np.float32)


    slices = 5
    y0,y1 = 100, 300
    x0, x1 = 400, 800
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
    fig.patch.set_facecolor('black')  # Set the background color to black
    im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,0].set_title('FBP, NA recon', color='red')
    axes[0,0].axis('off')
    im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,1].set_title('Old TV, NA recon', color='red')
    axes[0,1].axis('off')
    im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,0].axis('off')
    im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,1].axis('off')
    im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
    axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
    axes[2,0].axis('off')
    im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
    axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
    axes[2,1].axis('off')
    fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)

    slices = 5
    y0,y1 = 100, 330
    x0, x1 = 800, 1100
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
    fig.patch.set_facecolor('black')  # Set the background color to black
    im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,0].set_title('FBP, NA recon', color='red')
    axes[0,0].axis('off')
    im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
    axes[0,1].set_title('Old TV, NA recon', color='red')
    axes[0,1].axis('off')
    im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,0].axis('off')
    im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
    axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV), color='red')
    axes[1,1].axis('off')
    im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
    axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
    axes[2,0].axis('off')
    im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
    axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
    axes[2,1].axis('off')
    fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)

In [ ]:
alpha_sweep = [300, 1000]
eta_sweep = [0.03, 0.007, 0.001]
N_iter = 50
for eta in eta_sweep:
    grad_im = xi_vector_field(XA_, eta)
    for alpha_TV in alpha_sweep:

        G = alpha_TV * FGP_dTV(reference=XA_, eta=eta, device='gpu', nonnegativity=False)
        F = LeastSquares(A,b)
        reconstructor = FISTA(f=F, g=G, initial=initial)

        reconstructor.run(N_iter, verbose = 1)
        recon_slice_dTV = reconstructor.solution.copy().as_array().astype(np.float32)


        G = alpha_TV*FGP_TV(device='gpu',nonnegativity=False)
        F = LeastSquares(A,b)
        reconstructor = FISTA(f=F, g=G, initial=initial)

        reconstructor.run(N_iter, verbose = 1)
        recon_slice_TV = reconstructor.solution.copy().as_array().astype(np.float32)


        slices = 5
        y0,y1 = 100, 300
        x0, x1 = 400, 800
        fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
        fig.patch.set_facecolor('black')  # Set the background color to black
        im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
        axes[0,0].set_title('FBP, NA recon', color='red')
        axes[0,0].axis('off')
        im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
        axes[0,1].set_title('Old TV, NA recon', color='red')
        axes[0,1].axis('off')
        im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
        axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
        axes[1,0].axis('off')
        im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
        axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV) + 'eta: ' + str(eta), color='red')
        axes[1,1].axis('off')
        im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
        axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
        axes[2,0].axis('off')
        im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
        axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
        axes[2,1].axis('off')
        fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)

        slices = 5
        y0,y1 = 100, 330
        x0, x1 = 800, 1100
        fig, axes = plt.subplots(3, 2, figsize=(12, 10))  # 1 row, 3 columns
        fig.patch.set_facecolor('black')  # Set the background color to black
        im0 = axes[0,0].imshow(NA_fbp[slices,y0:y1, x0:x1],cmap='gray')
        axes[0,0].set_title('FBP, NA recon', color='red')
        axes[0,0].axis('off')
        im1 = axes[0,1].imshow(NA[slices,y0:y1, x0:x1],cmap='gray')
        axes[0,1].set_title('Old TV, NA recon', color='red')
        axes[0,1].axis('off')
        im2 = axes[1,0].imshow(recon_slice_TV[slices,y0:y1, x0:x1],cmap='gray')
        axes[1,0].set_title('TV, NA recon, alpha: ' + str(alpha_TV), color='red')
        axes[1,0].axis('off')
        im2 = axes[1,1].imshow(recon_slice_dTV[slices,y0:y1, x0:x1],cmap='gray')
        axes[1,1].set_title('dTV, NA recon, alpha: ' + str(alpha_TV) + 'eta: ' + str(eta), color='red')
        axes[1,1].axis('off')
        im2 = axes[2,0].imshow(XA[slices,y0:y1, x0:x1],cmap='gray')
        axes[2,0].set_title('TV, XA recon, alpha: ', color='red')
        axes[2,0].axis('off')
        im2 = axes[2,1].imshow(grad_im.as_array()[slices,y0:y1, x0:x1])
        axes[2,1].set_title('Gradient info, eta: ' + str(eta), color='red')
        axes[2,1].axis('off')
        fig.suptitle('dTV and TV comparisons, N_iterations: ' + str(N_iter), color='red', fontsize=16)

In [ ]:
A = range(1250,1260)
path = '/dtu-compute/msaca/sliceA_neutron_psi/output/dtv_recon/slice_dtv_####.tiff'
paths = ma.generate_paths(path,A)
ny, nx = np.shape(tifffile.imread(paths[0]))
dTV_recon = np.empty((len(A), ny, nx))
for i in range(len(A)):
    dTV_recon[i] = tifffile.imread(paths[i])

In [ ]:
slices = 5
y0,y1 = 100, 300
x0, x1 = 400, 800
plt.imshow(dTV_recon[slices,y0:y1, x0:x1], cmap = 'gray')